# Performance Metrics Comparison

**Objective:** Compare plate discipline, batted ball profiles, expected stats, and run expectancy across the three pitchers.

**Core Question:** Who actually prevents runs most effectively, and by how much?

In [ ]:
from analysis_utils import *
set_style()
dfs = load_all()
COLORS = {'Ohtani':'#1f77b4','Sanchez':'#ff7f0e','Misiorowski':'#2ca02c'}

# ---- Aggregate Summary Table ----
summary = build_summary_df(dfs)
key_cols = ['Pitcher','Pitches','Strike%','Zone%','Swing%','Whiff%','Chase%',
            'K%','BB%','K-BB%','AVG','OBP','SLG','OPS',
            'xBA','xwOBA','xSLG','BIP','Avg EV','HardHit%','Barrels','GB',
            'Total RE Delta','RE per Pitch']
print("COMPREHENSIVE SUMMARY TABLE")
print("=" * 140)
print(summary[key_cols].to_string(index=False))

In [ ]:
# ---- Plate Discipline ----
print("PLATE DISCIPLINE METRICS\n" + "="*70)
disc_rows = []
for name, df in dfs.items():
    d = plate_discipline(df)
    disc_rows.append({'Pitcher':name,'Swing%':f"{d['swing_pct']:.1f}%",
                      'Whiff%':f"{d['whiff_pct']:.1f}%",
                      'Contact%':f"{d['contact_pct']:.1f}%",
                      'Chase%':f"{d['chase_pct']:.1f}%",
                      'Zone%':f"{d['zone_pct']:.1f}%"})
print(pd.DataFrame(disc_rows).to_string(index=False))

print("\n\nKEY INSIGHT:")
print("Ohtani has the highest Zone% (50.3%) yet the LOWEST Chase% (29.8%).")
print("This suggests batters pick up his pitches out of hand more easily.")
print("Sanchez (Chase% 38.2%) generates more chases despite fewer strikes in the zone.")

In [ ]:
# ---- Statistical Significance Tests ----
from scipy.stats import chi2_contingency
ohtani = dfs['Ohtani']; sanchez = dfs['Sanchez']; misio = dfs['Misiorowski']

def compare_kbb(p1, p2, metric):
    p1_pa = p1[p1['events'].notna() & (p1['events'] != '')]
    p2_pa = p2[p2['events'].notna() & (p2['events'] != '')]
    p1_e = len(p1_pa[p1_pa['events']==metric])
    p2_e = len(p2_pa[p2_pa['events']==metric])
    _, p, _, _ = chi2_contingency([[p1_e, len(p1_pa)-p1_e], [p2_e, len(p2_pa)-p2_e]])
    return p

p_bb = compare_kbb(ohtani, sanchez, 'walk')
p_k = compare_kbb(ohtani, misio, 'strikeout')

o_bip = ohtani[ohtani['description']=='hit_into_play']
s_bip = sanchez[sanchez['description']=='hit_into_play']
o_gb = len(o_bip[o_bip['bb_type']=='ground_ball'])
s_gb = len(s_bip[s_bip['bb_type']=='ground_ball'])
_, p_gb, _, _ = chi2_contingency([[o_gb, len(o_bip)-o_gb], [s_gb, len(s_bip)-s_gb]])

print("CHI-SQUARE SIGNIFICANCE TESTS")
print("="*55)
for label, p_val in [('BB%: Ohtani vs Sanchez', p_bb),
                      ('K%: Ohtani vs Misiorowski', p_k),
                      ('GB%: Ohtani vs Sanchez', p_gb)]:
    sig = '***' if p_val<0.001 else '**' if p_val<0.01 else '*' if p_val<0.05 else 'ns'
    print(f"  {label:35s}  p={p_val:.4f}  {sig}")
print("\nSignificance: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant")

In [ ]:
# ---- Batted Ball Profile ----
print("BATTED BALL COMPARISON\n" + "="*70)
for name, df in dfs.items():
    bb = batted_balls(df)
    print(f"\n{name}:")
    print(f"  BIP: {bb['bip']}  |  Avg EV: {bb['avg_ev']:.1f} mph  |  Avg LA: {bb['avg_la']:.1f} deg")
    print(f"  Hard Hit%: {bb['hard_hit_pct']:.1f}%  |  Barrels: {bb['barrels']}")
    print(f"  GB: {bb['gb']}  FB: {bb['fb']}  LD: {bb['ld']}")

print("\nKEY INSIGHT:")
print("Ohtani leads in contact suppression (0 barrels, lower HardHit%).")
print("BUT Sanchez's extreme GB rate changes the damage profile entirely.")
print("Ground balls rarely produce XBH - this is an intentional strategy.")

In [ ]:
# ---- Run Expectancy ----
print("RUN EXPECTANCY (delta_pitcher_run_exp)\n" + "="*70)
for name, df in dfs.items():
    re = run_expectancy(df)
    print(f"\n{name}:")
    print(f"  Total: {re['total_re']:+.3f} runs  |  Per pitch: {re['avg_re']:+.4f} runs")
    print(f"  Positive (bad): {re['pos_events']}  Negative (good): {re['neg_events']}")

print("\nCRITICAL FINDING:")
print("Sanchez (+0.0080 per pitch) gives up LESS THAN HALF the run value of Ohtani (+0.0158).")
print("Despite throwing 465 MORE pitches, Sanchez's total run cost is LOWER.")
print("This is the strongest evidence that Sanchez is the more effective pitcher.")

In [ ]:
# ---- Key Metrics Bar Chart ----
fig, ax = plt.subplots(figsize=(14, 6))
names = list(dfs.keys())
metrics = {}
for name, df in dfs.items():
    kb = k_bb_stats(df); bb = batted_balls(df); pdm = plate_discipline(df)
    es = expected_stats(df)
    metrics[name] = [kb.get('k_pct',0), kb.get('bb_pct',0),
        pdm['whiff_pct'], es.get('xba',0)*1000, es.get('xwoba',0)*1000,
        bb.get('hard_hit_pct',0)]
labels = ['K%','BB%','Whiff%','xBA*1000\n(lower=better)','xwOBA*1000\n(lower=better)','HardHit%\n(lower=better)']
x = np.arange(len(labels)); w = 0.25
for i, name in enumerate(names):
    vals = metrics[name]
    bars = ax.bar(x + i*w - w, vals, w, label=name, color=COLORS[name], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{v:.1f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Value"); ax.set_title("Key Performance Metrics", fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('figures/key_metrics.png', dpi=150, bbox_inches='tight')
plt.show()